In [1]:
import os
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.pipeline import Pipeline
from xgboost import XGBClassifier

In [2]:
print('Loading data...')
train = pd.read_csv('../data/raw_data/train.csv')
test = pd.read_csv('../data/raw_data/test.csv')

train['Will_Buy_EV'] = train['Will_Buy_EV'].map({'No': 0, 'Yes': 1})

X = train.drop(columns=['Will_Buy_EV', 'id'], errors='ignore')
y = train['Will_Buy_EV']
X_test = test.drop(columns=['id'], errors='ignore')
test_ids = test['id']

Loading data...


In [3]:
number_transformer = Pipeline(steps=[('scaler', StandardScaler())])
category_transformer = Pipeline(steps=[('onehot', OneHotEncoder(handle_unknown='ignore'))])

preprocessor = ColumnTransformer(
    transformers=[
        ('numerical', number_transformer, make_column_selector(dtype_include='number')),
        ('categorical', category_transformer, make_column_selector(dtype_include='object'))
    ],
    remainder='drop'
)

In [4]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
xgb_test_preds = np.zeros(len(X_test))
cv_scores = []

print('Training XGBoost...')
for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
    X_tr, y_tr = X.iloc[train_idx], y.iloc[train_idx]
    X_val, y_val = X.iloc[val_idx], y.iloc[val_idx]
    
    model = Pipeline([
        ('preprocessor', preprocessor),
        ('classifier', XGBClassifier(n_estimators=500, learning_rate=0.05, eval_metric='auc', random_state=42, n_jobs=-1))
    ])
    
    model.fit(X_tr, y_tr)
    val_preds = model.predict_proba(X_val)[:, 1]
    auc = roc_auc_score(y_val, val_preds)
    cv_scores.append(auc)
    print(f'Fold {fold+1} AUC: {auc:.5f}')
    
    xgb_test_preds += model.predict_proba(X_test)[:, 1] / skf.n_splits

print(f'\nMean AUC: {np.mean(cv_scores):.5f}')

Training XGBoost...
Fold 1 AUC: 0.94057
Fold 2 AUC: 0.94140
Fold 3 AUC: 0.94276
Fold 4 AUC: 0.94229
Fold 5 AUC: 0.94163

Mean AUC: 0.94173


In [5]:
# Save Submission
submission = pd.DataFrame({'id': test_ids, 'Will_Buy_EV': xgb_test_preds})
os.makedirs('../output', exist_ok=True)
sub_path = '../output/submission_xgb.csv'
submission.to_csv(sub_path, index=False)
print(f'Saved to {sub_path}')

Saved to ../output/submission_xgb.csv
